# M4 — Multi-GPU LBVH: Scaling Results

GPU: Quadro RTX 6000 (sm_75, Turing) × 4 on a single node.  
Pipeline: Morton → CUB sort → samplesort (Alltoallv) → Karras+refit → top-level merge.  
Correctness: root AABB bit-identical for P = 1, 2, 4 on bunny (4 968 triangles).

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.rcParams.update({
    "figure.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 13,
})

STAGES = ["A Morton", "B1 CUB sort", "B3 Alltoallv", "C Karras+refit", "D top-level"]
COLORS = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]

# ── Log parser ────────────────────────────────────────────────────────────
_STAGE_PATS = [
    ("A Morton",       r"A\s+Morton"),
    ("B1 CUB sort",    r"B1\s+local"),
    ("B3 Alltoallv",   r"B3\s+Alltoallv"),
    ("C Karras+refit", r"C\s+Karras"),
    ("D top-level",    r"D\s+top"),
    ("TOTAL",          r"TOTAL"),
]

def parse_scaling_log(path):
    """Return {P: {stage_name: mean_ms}} from a strong or weak scaling log."""
    text = Path(path).read_text()
    out = {}
    for m in re.finditer(r"--- P=(\d+) rank", text):
        P = int(m.group(1))
        start = m.end()
        nxt = re.search(r"--- P=\d+ rank", text[start:])
        block = text[start: start + (nxt.start() if nxt else len(text) - start)]
        timings = {}
        for line in block.splitlines():
            for name, pat in _STAGE_PATS:
                if re.search(pat, line):
                    nums = re.findall(r"\d+\.\d+", line)
                    if len(nums) >= 2:
                        timings[name] = float(nums[1])   # mean column
        if timings:
            out[P] = timings
    return out

def _latest(log_dir, pattern):
    files = sorted(Path(log_dir).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None

LOG_DIR = Path("../sbatch/logs")
strong_log = _latest(LOG_DIR, "strong_*.out")
weak_log   = _latest(LOG_DIR, "weak_*.out")
print(f"Strong log : {strong_log}")
print(f"Weak log   : {weak_log}")

## Strong Scaling — N = 50 M triangles, fixed

Stage A timer now covers H2D transfer + GPU kernel only (disk I/O moved before the barrier).

In [ ]:
_sr = parse_scaling_log(strong_log)
P   = sorted(_sr.keys())

strong       = {s: [_sr[p].get(s, 0.0) for p in P] for s in STAGES}
strong_total = [_sr[p].get("TOTAL", 0.0) for p in P]
strong_shards_p4 = []   # populated from log if needed

print("Strong totals (ms):", dict(zip(P, strong_total)))
print(f"P=4 speedup vs P=1 : {strong_total[0]/strong_total[-1]:.2f}×")
print(f"P=4 parallel eff.  : {strong_total[0]/(P[-1]*strong_total[-1])*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(P))
bottoms = np.zeros(len(P))
for stage, color in zip(STAGES, COLORS):
    ax.bar(x, strong[stage], bottom=bottoms, color=color, label=stage, width=0.5)
    bottoms += np.array(strong[stage])
ax.set_xticks(x); ax.set_xticklabels([f"P={p}" for p in P])
ax.set_ylabel("Wall time (ms)")
ax.set_title("Strong scaling — stage breakdown\nN = 50 M triangles")
ax.legend(loc="upper left", fontsize=11)

ax2 = axes[1]
ideal = [strong_total[0] / p for p in P]
ax2.plot(P, strong_total, "o-", color="#4e79a7", linewidth=2, markersize=8, label="Measured total")
ax2.plot(P, ideal,        "s--", color="#e15759", linewidth=2, markersize=8, label="Ideal (linear)")
ax2.set_xscale("log", base=2); ax2.set_yscale("log")
ax2.set_xticks(P); ax2.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax2.set_xlabel("Number of GPUs (P)"); ax2.set_ylabel("Total wall time (ms)")
ax2.set_title("Strong scaling — total time"); ax2.legend()

plt.tight_layout()
plt.savefig("../latex/m4_strong_scaling.png", bbox_inches="tight")
plt.show()

## Weak Scaling — 10 M triangles / rank, growing N

In [ ]:
_wr   = parse_scaling_log(weak_log)
P_w   = sorted(_wr.keys())
N_per_rank = 10_000_000

weak       = {s: [_wr[p].get(s, 0.0) for p in P_w] for s in STAGES}
weak_total = [_wr[p].get("TOTAL", 0.0) for p in P_w]
weak_N     = [p * N_per_rank for p in P_w]

efficiency = [weak_total[0] / t * 100 for t in weak_total]
print("Weak totals (ms):", dict(zip(P_w, weak_total)))
print("Weak efficiency (%):", dict(zip(P_w, [f"{e:.1f}" for e in efficiency])))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(P_w))
bottoms = np.zeros(len(P_w))
for stage, color in zip(STAGES, COLORS):
    ax.bar(x, weak[stage], bottom=bottoms, color=color, label=stage, width=0.5)
    bottoms += np.array(weak[stage])
ax.set_xticks(x)
ax.set_xticklabels([f"P={p}\n(N={int(n/1e6)}M)" for p, n in zip(P_w, weak_N)])
ax.set_ylabel("Wall time (ms)")
ax.set_title("Weak scaling — stage breakdown\n10 M triangles / rank")
ax.legend(loc="upper left", fontsize=11)

ax2 = axes[1]
ax2.plot(P_w, efficiency, "o-", color="#4e79a7", linewidth=2, markersize=8, label="Measured efficiency")
ax2.axhline(100, color="#e15759", linestyle="--", linewidth=2, label="Ideal (100 %)")
ax2.set_xscale("log", base=2)
ax2.set_xticks(P_w); ax2.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax2.set_ylim(0, 120)
ax2.set_xlabel("Number of GPUs (P)"); ax2.set_ylabel("Weak scaling efficiency (%)")
ax2.set_title("Weak scaling — efficiency\n(T₁ / T_P × 100 %)"); ax2.legend()

plt.tight_layout()
plt.savefig("../latex/m4_weak_scaling.png", bbox_inches="tight")
plt.show()

## Bottleneck breakdown — GPU compute vs communication

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, data, ps, total, title in [
    (axes[0], strong, P,   strong_total, "Strong scaling — N=50M"),
    (axes[1], weak,   P_w, weak_total,   "Weak scaling — 10M/rank"),
]:
    gpu_compute = [data["A Morton"][i] + data["B1 CUB sort"][i] + data["C Karras+refit"][i]
                   for i in range(len(ps))]
    mpi_comm    = [data["B3 Alltoallv"][i] + data["D top-level"][i]
                   for i in range(len(ps))]

    x = np.arange(len(ps))
    ax.bar(x, gpu_compute, color="#4e79a7", label="GPU compute (A+B1+C)", width=0.5)
    ax.bar(x, mpi_comm,    bottom=gpu_compute, color="#e15759", label="MPI comm (B3+D)", width=0.5)
    ax.set_xticks(x); ax.set_xticklabels([f"P={p}" for p in ps])
    ax.set_ylabel("Wall time (ms)"); ax.set_title(title); ax.legend(fontsize=9)

    top = max(g + m for g, m in zip(gpu_compute, mpi_comm))
    ax.set_ylim(0, top * 1.20)
    for i, (g, m) in enumerate(zip(gpu_compute, mpi_comm)):
        pct = m / (g + m) * 100 if (g + m) > 0 else 0
        if m > top * 0.03:
            ax.text(i, g + m * 0.55, f"{pct:.0f}%\ncomm",
                    ha="center", va="center", fontsize=10, color="white", fontweight="bold")
        else:
            ax.text(i, g + m + top * 0.02, f"{pct:.0f}%\ncomm",
                    ha="center", va="bottom", fontsize=10, color="#e15759")

plt.tight_layout()
plt.savefig("../latex/m4_compute_vs_comm.png", bbox_inches="tight")
plt.show()